In [263]:
import pandas as pd
from pandas import DataFrame
import numpy as np
import networkx as nx
import peartree as pear
import partridge as ptg
import matplotlib.pyplot as plt
import itertools
import warnings
warnings.filterwarnings('ignore')

In [264]:
feed_all = pear.get_representative_feed("gtfs_subway/")
feed = feed_all

In [265]:
def iterator(L):
	a, b = itertools.tee(L)
	next(b, None)
	return list(zip(a,b))

In [266]:
G = nx.MultiGraph(name="midday")

In [267]:
station_data = pd.read_csv("data/MTA_Subway_Stations.csv")
station_data = station_data[['GTFS Stop ID', 'Complex ID', 'Stop Name', 'Daytime Routes']]

rl = [str(r).split() for r in station_data['Daytime Routes']]
station_data['Daytime Routes'] = rl

# remove Staten Island Railroad

station_data = station_data[station_data['Daytime Routes'].apply(lambda x: "SIR" not in x)]
station_data.set_index('GTFS Stop ID', drop=False, inplace=True)
nodes_list = [(k, v) for k,v in station_data.to_dict('index').items()]

In [268]:
G.add_nodes_from(nodes_list)

In [269]:
line_names = [str(s) for s in [1,2,3,4,5,6,'6X',7,'7X','A','B','C','D','E','F','G','H','J','L','M','N','Q','R','W','Z','FS','GS','Transfer']]

In [270]:
## transfers in complex
transfers = pd.read_csv('gtfs_subway/transfers.txt')
	
transfers.drop("transfer_type", inplace=True, axis=1)

transfers = transfers[transfers['from_stop_id'] != transfers['to_stop_id']]
transfers.reset_index(inplace=True,drop=True)
transfers

,from_stop_id,to_stop_id,min_transfer_time
0,112,A09,180
1,125,A24,180
2,127,725,180
3,127,902,180
4,127,A27,300
...,...,...,...
147,R31,235,180
148,R31,D24,300
149,R33,F23,180
150,S01,A45,180


In [271]:
complexes = pd.read_csv("data/MTA_Subway_Stations_and_Complexes_20250918.csv")
complexes = complexes[complexes['Number Of Stations In Complex'] > 1]
complexes = complexes[['Complex ID','Number Of Stations In Complex', 'GTFS Stop IDs']]
gtfs_ids = [ids.split("; ") for ids in complexes['GTFS Stop IDs']]
complex_pairs = [list(itertools.combinations(sts, 2)) for sts in gtfs_ids]
complexes['complex_transfers'] = complex_pairs
complexes = complexes.explode('complex_transfers', ignore_index=True).drop(['Number Of Stations In Complex','GTFS Stop IDs'], axis= 1)
transfer_edges = list(zip(transfers['from_stop_id'], transfers['to_stop_id']))
transfer_edge_set = set()
for k in transfer_edges:
	transfer_edge_set.add(tuple(sorted(k)))

##print(transfer_edge_set ^ set(complexes['complex_transfers']))
##('254', 'L26'), ('629', 'B08'), ('B08', 'R11')##

## Out of station transfers specified on the official map
## Walk Junius to Livonia
## Walk Lexington-59 NRW to Lexington-63 -- 300s
## walk Lexington-59 456 to Lexington-63 -- 300s
transfers['edges'] = [tuple(sorted(k)) for k in transfer_edges]
unique_transfers = transfers.sort_values('edges').iloc[::2,:]
unique_transfers.reset_index(drop=True,inplace=True)
unique_transfers.drop(['from_stop_id', 'to_stop_id'], axis=1, inplace=True)

In [311]:
unique_transfers.edges.to_csv('transfer_edges.csv')

In [272]:
fstop = [unique_transfers['edges'][k][0] for k in unique_transfers.index]
tstop = [unique_transfers['edges'][k][1] for k in unique_transfers.index]
ttime = unique_transfers['min_transfer_time']
transfer_edge_weights = {(a, b): {"Transfer": t} for a,b,t in list(zip(fstop, tstop, ttime))}

In [273]:
el = list(G.edges(data=True))

for i in range(len(el)):
	q = (el[i][0], el[i][1])
	d = el[i][2]
	el[i] = [q,d]

edf = pd.DataFrame([k[1] for k in el], index= [k[0] for k in el])
ndf = pd.DataFrame.from_dict(dict(G.nodes(data=True))).T


### Fix edge weights so that boarding cost varies by line

In [274]:
## midday 9:30 am to 3:30 pm
start_time = int(9.5 * 60 * 60)
end_time = int(15.5 * 60 * 60)

In [275]:
start_time = int(9.5 * 60 * 60)
end_time = int(15.5 * 60 * 60)
tripline = list(feed.stop_times['trip_id'])
inseq = feed.stop_times.copy()
tripdict = {k: v for k,v in zip(feed.trips['trip_id'], feed.trips['route_id'])}
feed.stop_times['lines'] = [tripdict[s] for s in feed.stop_times['trip_id']]
feed.stop_times['dir'] = [s[-4] if s[-4] != 'X' else s[-7] for s in feed.stop_times['trip_id']]
stoptime = pd.DataFrame(feed.stop_times).copy()
stoptime = stoptime[stoptime['arrival_time'].between(start_time, end_time)]
stoptime = stoptime[stoptime['lines'] != 'SI']
stoptime.sort_values(['stop_id','lines', 'dir', 'arrival_time'], inplace=True)

In [276]:
seq_dict = {}
for n, g in inseq.groupby('trip_id'):
	seq_dict[n] = len(g['stop_sequence'])

In [277]:
G.nodes['713']

{'GTFS Stop ID': '713',
 'Complex ID': 457,
 'Stop Name': '52 St',
 'Daytime Routes': ['7']}

In [278]:
stoptime.sort_values('trip_id', inplace=True)
stoptime['trip_len'] = [seq_dict[n] for n in stoptime['trip_id']]
stops_in_time = dict(stoptime.trip_id.value_counts())
stoptime['count_in_timespan'] = [stops_in_time[n] for n in stoptime.trip_id]
stoptime = stoptime[stoptime['count_in_timespan'] == stoptime['trip_len']]

In [279]:
stgroups = stoptime.groupby(['stop_id', 'lines','dir'], group_keys=True)
st_weights = pd.DataFrame(list(stgroups.groups.keys()),index=list(stgroups.groups.keys()), columns=['stop', 'line', 'dir']).reset_index(drop=False)
st_weights['weight'] = 0
i = 0
for name, group in stgroups:
	waittime = iterator(list(group['arrival_time']))
	dirw = round(np.mean([b-a for a,b in waittime]), 3)
	st_weights.at[i, 'weight'] = dirw
	i += 1


st_weights = st_weights.groupby(['stop','line'], as_index=False).mean('weight')
st_weights.sort_values('stop', inplace=True)

node_weight_dict = {}
for name, group in st_weights.groupby(['stop'],group_keys=True):
	float_w = dict(zip(group['line'], group['weight']))
	float_w = {k: round(v,3) for k,v in float_w.items()}
	node_weight_dict[name[0]] = float_w

In [280]:
st_weights

,stop,line,weight
0,101,1,333.8755
1,103,1,334.9665
2,104,1,334.9665
3,106,1,334.9665
4,107,1,334.9665
...,...,...,...
753,R44,R,479.5455
754,R45,R,479.5455
755,S01,FS,600.0000
756,S03,FS,600.0000


In [281]:
G.nodes['713']

{'GTFS Stop ID': '713',
 'Complex ID': 457,
 'Stop Name': '52 St',
 'Daytime Routes': ['7']}

In [282]:
for n in G.nodes:
	G.nodes[n]['wait_time'] = node_weight_dict[n]
	G.nodes[n]['lines'] = set(node_weight_dict[n].keys())
	try:
		del G.nodes[n]['Daytime Routes']
	except KeyError:
		continue

In [283]:
dict_7x = {k: v for k,v in node_weight_dict.items() if '7X' in v}
nan7x = {k: v for k,v in dict_7x.items() if np.isnan(v.get('7X'))}


In [284]:
## fix nan for one-off 7X routes in midday
G.nodes['710']['wait_time']['7X'] = 495.0
G.nodes['711']['wait_time']['7X'] = 495.0
G.nodes['713']['wait_time']['7X'] = 465.0

## remove the single 5 train to gun hill rd in non-rushhour

try:
	del G.nodes['208']['wait_time']['5']
except:
	pass

In [285]:
for n in G.nodes:
	G.nodes[n]['lines'] = set(G.nodes[n]['wait_time'].keys())

In [286]:
## do edges

stops_per_line = stoptime.sort_values(['lines','trip_len'])
stops_per_line = stops_per_line.groupby('trip_id', group_keys=False).apply(lambda x: x.sort_values('stop_sequence'))


In [287]:
edge_set = set()

In [288]:
for name, group in stops_per_line.groupby('trip_id'):
	edges = iterator(group.stop_id.to_list())
	for k in edges:
		edge_set.add(tuple(sorted(k)))
edge_set

{('101', '103'),
 ('103', '104'),
 ('104', '106'),
 ('106', '107'),
 ('107', '108'),
 ('108', '109'),
 ('109', '110'),
 ('110', '111'),
 ('111', '112'),
 ('112', '113'),
 ('113', '114'),
 ('114', '115'),
 ('115', '116'),
 ('116', '117'),
 ('117', '118'),
 ('118', '119'),
 ('119', '120'),
 ('120', '121'),
 ('120', '123'),
 ('120', '227'),
 ('121', '122'),
 ('122', '123'),
 ('123', '124'),
 ('123', '127'),
 ('124', '125'),
 ('125', '126'),
 ('126', '127'),
 ('127', '128'),
 ('128', '129'),
 ('128', '132'),
 ('129', '130'),
 ('130', '131'),
 ('131', '132'),
 ('132', '133'),
 ('132', '137'),
 ('133', '134'),
 ('134', '135'),
 ('135', '136'),
 ('136', '137'),
 ('137', '138'),
 ('137', '228'),
 ('138', '139'),
 ('139', '142'),
 ('201', '204'),
 ('204', '205'),
 ('205', '206'),
 ('206', '207'),
 ('207', '208'),
 ('208', '209'),
 ('208', '213'),
 ('209', '210'),
 ('210', '211'),
 ('211', '212'),
 ('212', '213'),
 ('213', '214'),
 ('213', '221'),
 ('213', '505'),
 ('214', '215'),
 ('215', '216'

In [ ]:
w = {e: {} for e in edge_set}
direction = {e: '' for e in edge_set}
stoptime = stoptime.groupby('trip_id', as_index=False, group_keys=False).apply(lambda x: x.sort_values('stop_sequence', ascending=True))
for name, group in stoptime.groupby('trip_id'):
	line = group.lines.values[0]
	delta = group['arrival_time'].diff()
	stop = list(zip(list(zip(group['stop_id'], group['stop_id'].shift())), delta))[1:]
	for e,t in stop:
		e = tuple(sorted(list(e)))
		try:
			w[e][line]
		except KeyError:
			w[e][line] = []
		finally:
			w[e][line].append(t)


In [291]:
def roundlist(x):
	if type(x) == list: return int(np.ceil(np.mean(x)))
	else: return x

In [292]:
all_edge_times = pd.DataFrame.from_dict(w).T
avg_times = all_edge_times.copy()
meandf = avg_times.map(lambda x: roundlist(x))
edge_weights = meandf.apply(lambda x: x.dropna().to_dict(),axis=1).to_dict()

In [293]:
def enum(x):
	i = range(len(x))
	l = [list(k) for k in zip(x, i)]
	return l

def enumone(x):
	i = range(1, len(x)+1)
	l = [list(k) for k in zip(x, i)]
	return l

In [294]:
def intersect(x,y):
	return(set(x) & set(y))

def set_diff(x,y): return(set(x) - set(y))

In [295]:
edge_weights = edge_weights | transfer_edge_weights

In [296]:
line_order = [str(s) for s in [1,2,3,4,5,6,'6X',7,'7X','A','B','C','D','E','F','G','H','J','L','M','N','Q','R','W','Z','FS','GS','Transfer']]

In [297]:
exp = pd.DataFrame(edge_weights.items(), columns=['stops', 'time'])
exp.set_index("stops", inplace=True)
exp['time'] = [list(k.items()) for k in exp['time']]
exp = exp.explode('time').reset_index()
exp['line'] = [t[0] for t in exp.time]
exp['time'] = [t[1] for t in exp.time]
exp = exp[['stops', 'line', 'time']]
exp['order'] = [line_order.index(x) for x in exp['line']]
exp = exp.sort_values('order').drop('order', axis=1)

In [298]:
exp['edge_direction'] = [direction.get(w, 'T') for w in exp['stops']]

In [299]:
exp

,stops,line,time,edge_direction
41,"(136, 137)",1,94.0,N
367,"(121, 122)",1,76.0,N
680,"(119, 120)",1,105.0,N
679,"(139, 142)",1,105.0,N
342,"(107, 108)",1,76.0,N
...,...,...,...,...
772,"(635, R20)",Transfer,180.0,T
771,"(635, L03)",Transfer,180.0,T
770,"(631, 901)",Transfer,180.0,T
777,"(640, M21)",Transfer,180.0,T


In [300]:
exp.to_csv("edges.csv")

In [301]:
zipkey = zip(exp.stops, exp.line)
keyed = [(i[0],i[1],j) for i,j in zipkey]
edge_dict = dict({k: v for k,v in zip(keyed, exp.time)})
edge_dict

{('136', '137', '1'): 94.0,
 ('121', '122', '1'): 76.0,
 ('119', '120', '1'): 105.0,
 ('139', '142', '1'): 105.0,
 ('107', '108', '1'): 76.0,
 ('128', '129', '1'): 90.0,
 ('112', '113', '1'): 111.0,
 ('134', '135', '1'): 90.0,
 ('108', '109', '1'): 90.0,
 ('130', '131', '1'): 60.0,
 ('120', '121', '1'): 120.0,
 ('114', '115', '1'): 90.0,
 ('101', '103', '1'): 206.0,
 ('118', '119', '1'): 60.0,
 ('123', '124', '1'): 90.0,
 ('129', '130', '1'): 60.0,
 ('117', '118', '1'): 60.0,
 ('115', '116', '1'): 111.0,
 ('137', '138', '1'): 67.0,
 ('122', '123', '1'): 90.0,
 ('125', '126', '1'): 120.0,
 ('138', '139', '1'): 90.0,
 ('113', '114', '1'): 120.0,
 ('106', '107', '1'): 90.0,
 ('103', '104', '1'): 90.0,
 ('104', '106', '1'): 90.0,
 ('124', '125', '1'): 90.0,
 ('111', '112', '1'): 120.0,
 ('109', '110', '1'): 112.0,
 ('132', '133', '1'): 90.0,
 ('126', '127', '1'): 106.0,
 ('135', '136', '1'): 60.0,
 ('127', '128', '1'): 90.0,
 ('131', '132', '1'): 60.0,
 ('116', '117', '1'): 150.0,
 ('110',

In [302]:
G_adjusted = G.copy()
G_adjusted.clear_edges()
for u,v,k in edge_dict.keys():
	G_adjusted.add_edge(u,v,key=k, travel_time=edge_dict[(u,v,k)])

In [303]:
G_adjusted.get_edge_data('226','227')

{'2': {'travel_time': 60.0}, '3': {'travel_time': 60.0}}

In [304]:
G_adjusted.edges(data=True, keys=True)

MultiEdgeDataView([('127', '126', '1', {'travel_time': 106.0}), ('127', '128', '1', {'travel_time': 90.0}), ('127', '128', '2', {'travel_time': 60.0}), ('127', '128', '3', {'travel_time': 60.0}), ('127', '123', '2', {'travel_time': 240.0}), ('127', '123', '3', {'travel_time': 241.0}), ('127', '725', 'Transfer', {'travel_time': 180.0}), ('127', '902', 'Transfer', {'travel_time': 180.0}), ('127', 'A27', 'Transfer', {'travel_time': 300.0}), ('127', 'R16', 'Transfer', {'travel_time': 180.0}), ('S01', 'S03', 'FS', {'travel_time': 120.0}), ('S01', 'A45', 'Transfer', {'travel_time': 180.0}), ('254', '253', '3', {'travel_time': 75.0}), ('254', '255', '3', {'travel_time': 90.0}), ('254', 'L26', 'Transfer', {'travel_time': 300.0}), ('M01', 'M04', 'M', {'travel_time': 150.0}), ('726', '725', '7', {'travel_time': 166.0}), ('726', '725', '7X', {'travel_time': 195.0}), ('713', '714', '7', {'travel_time': 60.0}), ('713', '714', '7X', {'travel_time': 60.0}), ('713', '712', '7', {'travel_time': 120.0})

In [305]:
# fix edge to gun hill road
G_adjusted.remove_edge('208', '213')

In [306]:
G_adjusted['710']

AdjacencyView({'709': {'7': {'travel_time': 90.0}}, '711': {'7': {'travel_time': 61.0}, '7X': {'travel_time': 90.0}}, '712': {'7': {'travel_time': 150.0}}, '707': {'7X': {'travel_time': 180.0}}, 'G14': {'Transfer': {'travel_time': 180.0}}})

In [307]:
ndfa = pd.DataFrame(dict(G_adjusted.nodes.data())).T
ndfa.drop('lines', axis=1, inplace=True)
ndfa['wait_time'] = [[[k,v] for k,v in q.items()] for q in ndfa['wait_time']]
ndfa = ndfa.explode('wait_time').reset_index(drop=True)
ndfa = pd.concat([ndfa, pd.DataFrame(list(ndfa['wait_time']), columns=['line', 'time'])], axis=1).drop('wait_time', axis=1)
ndfa = ndfa.sort_values('time')
ndfa

,GTFS Stop ID,Complex ID,Stop Name,line,time
585,L13,124,Montrose Av,L,265.068
498,L06,119,1 Av,L,265.068
508,L02,601,6 Av,L,265.068
441,L12,123,Grand St,L,265.068
297,L05,118,3 Av,L,265.068
...,...,...,...,...,...
324,H07,205,Beach 60 St,A,1004.000
539,H04,199,Broad Channel,A,1008.285
458,H03,198,Howard Beach-JFK Airport,A,1008.285
246,H02,197,Aqueduct-N Conduit Av,A,1008.285


## export

In [308]:
import json

def serialize(obj):
    if isinstance(obj, np.int64):
         return int(obj)
    if not isinstance(obj, (str, int, float, bool)):
        return list(obj)
    else: return obj

G_adjusted.name = "NYC Subway_multi-edge"
try: del G_adjusted.graph['crs']
except KeyError:
	pass
data1 = nx.node_link_data(G_adjusted, edges="edges")
with open('subwaygraph_midday.json', 'w') as f:
    json.dump(data1,f, default=serialize)